In [ ]:
# Installs and updates the runtime dependencies required for pose estimation and video processing.
!pip -q install -U ultralytics opencv-python numpy pandas ipywidgets ipyevents pillow

In [ ]:
# Imports required packages to support reproducible data processing and model inference.
import os
# Imports required packages to support reproducible data processing and model inference.
import io
# Imports required packages to support reproducible data processing and model inference.
import time
# Imports required packages to support reproducible data processing and model inference.
import json
# Imports required packages to support reproducible data processing and model inference.
import math
# Imports required packages to support reproducible data processing and model inference.
import subprocess
# Imports required packages to support reproducible data processing and model inference.
from pathlib import Path

# Imports required packages to support reproducible data processing and model inference.
import numpy as np
# Imports required packages to support reproducible data processing and model inference.
import pandas as pd
# Imports required packages to support reproducible data processing and model inference.
import cv2
# Imports required packages to support reproducible data processing and model inference.
from PIL import Image

# Imports required packages to support reproducible data processing and model inference.
from ultralytics import YOLO

# Imports required packages to support reproducible data processing and model inference.
import ipywidgets as widgets

# Imports required packages to support reproducible data processing and model inference.
from ipyevents import Event
# Imports required packages to support reproducible data processing and model inference.
from IPython.display import display, Video, HTML

# Defines run_cmd to encapsulate reusable logic for the pipeline.
def run_cmd(cmd):
    """Run a shell command safely and print stdout/stderr on failure."""
    # Implements error handling to surface failures with actionable diagnostics.
    try:
        # Executes this step as part of the end to end video pose analytics workflow.
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
        # Returns the computed value to the caller for downstream use.
        return out
    # Implements error handling to surface failures with actionable diagnostics.
    except subprocess.CalledProcessError as e:
        # Logs key intermediate outputs to support debugging and auditability.
        print("Command failed:", " ".join(cmd))
        # Logs key intermediate outputs to support debugging and auditability.
        print(e.output)
        # Executes this step as part of the end to end video pose analytics workflow.
        raise


In [ ]:










# Imports required packages to support reproducible data processing and model inference.
import os
# Imports required packages to support reproducible data processing and model inference.
import shlex
# Imports required packages to support reproducible data processing and model inference.
import subprocess

# Stores a configuration constant to standardize inputs and outputs across runs.
BUCKET = "msads-mba-capstone-team-2"

# Stores a configuration constant to standardize inputs and outputs across runs.
AM_VIDEO = "data/amateur/YouTube_3-0-Pickleball-Match.mp4"
# Stores a configuration constant to standardize inputs and outputs across runs.
PRO_VIDEO = "data/pro/YouTube_MEN-S-PRO-GOLD-2024-US-Open-Pickleball.mp4"

# Stores a configuration constant to standardize inputs and outputs across runs.
RUN_DIR = "runs_ready_score"
# Creates the local output directory to ensure downstream file writes succeed.
os.makedirs(RUN_DIR, exist_ok=True)

# Constructs a platform safe file path for intermediate artifacts.
LOCAL_AM_VIDEO = os.path.join(RUN_DIR, "amateur_source.mp4")

# Constructs a platform safe file path for intermediate artifacts.
LOCAL_PRO_VIDEO = os.path.join(RUN_DIR, "pro_source.mp4")

# Constructs a platform safe file path for intermediate artifacts.
AM_CLIP = os.path.join(RUN_DIR, "amateur_clip1.mp4")
# Constructs a platform safe file path for intermediate artifacts.
AM_CLIP_2 = os.path.join(RUN_DIR, "amateur_clip2.mp4")
# Constructs a platform safe file path for intermediate artifacts.
AM_CLIP_3 = os.path.join(RUN_DIR, "amateur_clip3.mp4")

# Constructs a platform safe file path for intermediate artifacts.
PRO_CLIP = os.path.join(RUN_DIR, "pro_clip1.mp4")
# Constructs a platform safe file path for intermediate artifacts.
PRO_CLIP_2 = os.path.join(RUN_DIR, "pro_clip2.mp4")
# Constructs a platform safe file path for intermediate artifacts.
PRO_CLIP_3 = os.path.join(RUN_DIR, "pro_clip3.mp4")

# Stores a configuration constant to standardize inputs and outputs across runs.
AM_START = "00:06:40"
# Stores a configuration constant to standardize inputs and outputs across runs.
AM_START_2 = "00:07:35"
# Stores a configuration constant to standardize inputs and outputs across runs.
AM_START_3 = "00:13:38"

# Stores a configuration constant to standardize inputs and outputs across runs.
PRO_START = "00:33:06"
# Stores a configuration constant to standardize inputs and outputs across runs.
PRO_START_2 = "00:19:18"
# Stores a configuration constant to standardize inputs and outputs across runs.
PRO_START_3 = "00:25:00"

# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
DUR_S = 10
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
TARGET_FPS = 30
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
TARGET_W = 1280


# Defines run_cmd to encapsulate reusable logic for the pipeline.
def run_cmd(cmd):
"""Run a shell command safely and print stdout/stderr on failure."""
# Implements error handling to surface failures with actionable diagnostics.
try:
    # Executes this step as part of the end to end video pose analytics workflow.
    out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
    # Returns the computed value to the caller for downstream use.
    return out
# Implements error handling to surface failures with actionable diagnostics.
except subprocess.CalledProcessError as e:










    # Logs key intermediate outputs to support debugging and auditability.
    print("Command failed:", " ".join(shlex.quote(c) for c in cmd))
    # Logs key intermediate outputs to support debugging and auditability.
    print(e.output)
    # Executes this step as part of the end to end video pose analytics workflow.
    raise

# Defines _as_gs_uri to encapsulate reusable logic for the pipeline.
def _as_gs_uri(p: str) -> str:
"""Convert relative bucket path -> gs://BUCKET/... (leave gs:// as-is)."""
# Applies a guard condition to handle missing data and enforce expected preconditions.
if p.startswith("gs://"):
    # Returns the computed value to the caller for downstream use.
    return p
# Executes this step as part of the end to end video pose analytics workflow.
p2 = p.lstrip("/") # normalize
# Returns the computed value to the caller for downstream use.
return f"gs://{BUCKET}/{p2}"

# Defines ensure_local_video to encapsulate reusable logic for the pipeline.
def ensure_local_video(src_path: str, local_path: str) -> str:
"""
Ensures a playable local MP4 exists at local_path.
- If src_path exists locally, use it.
- Else try copying from GCS using BUCKET + provided relative path.

- If that fails, try a fallback using the same filename under Data/... folders
(useful if objects actually live in Data/Amateur Videos/ or Data/
Professional Videos/).
"""
# Applies a guard condition to handle missing data and enforce expected preconditions.
if os.path.exists(src_path):
    # Returns the computed value to the caller for downstream use.
    return src_path
# Applies a guard condition to handle missing data and enforce expected preconditions.
if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
    # Logs key intermediate outputs to support debugging and auditability.
    print(f"Local file already exists: {local_path}")
    # Returns the computed value to the caller for downstream use.
    return local_path
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
candidates = []


# Executes this step as part of the end to end video pose analytics workflow.
fname = os.path.basename(src_path)
# Executes this step as part of the end to end video pose analytics workflow.
candidates.append(f"gs://{BUCKET}/Data/Amateur Videos/{fname}")
# Executes this step as part of the end to end video pose analytics workflow.
candidates.append(f"gs://{BUCKET}/Data/Professional Videos/{fname}")
# Executes this step as part of the end to end video pose analytics workflow.
last_err = None
# Iterates through streamed results to accumulate frame level metrics.
for gs_uri in candidates:
    # Implements error handling to surface failures with actionable diagnostics.
    try:
        # Logs key intermediate outputs to support debugging and auditability.
        print(f"Copying from GCS -> local: {gs_uri} -> {local_path}")
        # Executes an external command to perform system level media or data operations.
        run_cmd(["gsutil", "-m", "cp", gs_uri, local_path])










        # Applies a guard condition to handle missing data and enforce expected preconditions.
        if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        # Returns the computed value to the caller for downstream use.
        return local_path
    # Implements error handling to surface failures with actionable diagnostics.
    except Exception as e:
        # Executes this step as part of the end to end video pose analytics workflow.
        last_err = e
        # Executes this step as part of the end to end video pose analytics workflow.
        continue

# Executes this step as part of the end to end video pose analytics workflow.
raise FileNotFoundError(
    # Executes this step as part of the end to end video pose analytics workflow.
    "Could not find/copy the video from any of these GCS locations:\n"
    # Executes this step as part of the end to end video pose analytics workflow.
    + "\n".join(f" - {c}" for c in candidates)
    # Executes this step as part of the end to end video pose analytics workflow.
    + "\n\nIf gsutil is authenticated, one of those paths should work. "
    # Executes this step as part of the end to end video pose analytics workflow.
    "If not, run: `gcloud auth login` (or use the notebook’s auth flow)."
# Executes this step as part of the end to end video pose analytics workflow.
) from last_err

# Executes this step as part of the end to end video pose analytics workflow.
AM_VIDEO_LOCAL = ensure_local_video(AM_VIDEO, LOCAL_AM_VIDEO)
# Executes this step as part of the end to end video pose analytics workflow.
PRO_VIDEO_LOCAL = ensure_local_video(PRO_VIDEO, LOCAL_PRO_VIDEO)


# Logs key intermediate outputs to support debugging and auditability.
print("AM_VIDEO_LOCAL:", AM_VIDEO_LOCAL)
# Logs key intermediate outputs to support debugging and auditability.
print("PRO_VIDEO_LOCAL:", PRO_VIDEO_LOCAL)
# Executes this step as part of the end to end video pose analytics workflow.
AM_VIDEO_LOCAL: data/amateur/YouTube_3-0-Pickleball-Match.mp4
# Executes this step as part of the end to end video pose analytics workflow.
PRO_VIDEO_LOCAL: data/pro/YouTube_MEN-S-PRO-GOLD-2024-US-Open-Pickleball.mp4


In [ ]:
# Imports required packages to support reproducible data processing and model inference.
import os

# Defines extract_10s_clip to encapsulate reusable logic for the pipeline.
def extract_10s_clip(src, start_ts, out_path, duration=DUR_S, fps=TARGET_FPS, width=TARGET_W):
    """
    Extract a normalized H.264 MP4 clip for inference.
    start_ts can be "HH:MM:SS" (recommended) or seconds as float/int.
    """
    # Applies a guard condition to handle missing data and enforce expected preconditions.
    if os.path.exists(out_path):
        # Executes this step as part of the end to end video pose analytics workflow.
        os.remove(out_path)

    # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
    cmd = [
        # Builds an ffmpeg command that standardizes clip duration, frame rate, and resolution for inference.
        "ffmpeg", "-hide_banner", "-y",
        # Executes this step as part of the end to end video pose analytics workflow.
        "-ss", str(start_ts),
        # Executes this step as part of the end to end video pose analytics workflow.
        "-i", src,
        # Executes this step as part of the end to end video pose analytics workflow.
        "-t", str(float(duration)),
        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        "-vf", f"fps={int(fps)},scale={int(width)}:-2",
        # Executes this step as part of the end to end video pose analytics workflow.
        "-c:v", "libx264",
        # Executes this step as part of the end to end video pose analytics workflow.
        "-pix_fmt", "yuv420p",
        # Executes this step as part of the end to end video pose analytics workflow.
        "-movflags", "+faststart",
        # Executes this step as part of the end to end video pose analytics workflow.
        "-an",
        # Executes this step as part of the end to end video pose analytics workflow.
        out_path











    # Executes this step as part of the end to end video pose analytics workflow.
    ]
    # Executes an external command to perform system level media or data operations.
    run_cmd(cmd)

# Executes this step as part of the end to end video pose analytics workflow.
extract_10s_clip(AM_VIDEO_LOCAL, AM_START, AM_CLIP)
# Executes this step as part of the end to end video pose analytics workflow.
extract_10s_clip(AM_VIDEO_LOCAL, AM_START_2, AM_CLIP_2)
# Executes this step as part of the end to end video pose analytics workflow.
extract_10s_clip(AM_VIDEO_LOCAL, AM_START_3, AM_CLIP_3)

# Executes this step as part of the end to end video pose analytics workflow.
extract_10s_clip(PRO_VIDEO_LOCAL, PRO_START, PRO_CLIP)
# Executes this step as part of the end to end video pose analytics workflow.
extract_10s_clip(PRO_VIDEO_LOCAL, PRO_START_2, PRO_CLIP_2)
# Executes this step as part of the end to end video pose analytics workflow.
extract_10s_clip(PRO_VIDEO_LOCAL, PRO_START_3, PRO_CLIP_3)

# Logs key intermediate outputs to support debugging and auditability.
print("Wrote clips:")
# Logs key intermediate outputs to support debugging and auditability.
print(" ", AM_CLIP)
# Logs key intermediate outputs to support debugging and auditability.
print(" ", AM_CLIP_2)

# Logs key intermediate outputs to support debugging and auditability.
print(" ", AM_CLIP_3)
# Logs key intermediate outputs to support debugging and auditability.
print(" ", PRO_CLIP)
# Logs key intermediate outputs to support debugging and auditability.
print(" ", PRO_CLIP_2)
# Logs key intermediate outputs to support debugging and auditability.
print(" ", PRO_CLIP_3)

# Executes this step as part of the end to end video pose analytics workflow.
Wrote clips:
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/amateur_clip1.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/amateur_clip2.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/amateur_clip3.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/pro_clip1.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/pro_clip2.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/pro_clip3.mp4

In [ ]:
        # Defines _dist to encapsulate reusable logic for the pipeline.
        def _dist(a, b):
# Returns the computed value to the caller for downstream use.
return float(np.linalg.norm(a - b))

        # Defines _angle to encapsulate reusable logic for the pipeline.
        def _angle(a, b, c):
"""Angle ABC in degrees (at b)."""
# Executes this step as part of the end to end video pose analytics workflow.
ba = a - b
# Executes this step as part of the end to end video pose analytics workflow.
bc = c - b
# Executes this step as part of the end to end video pose analytics workflow.
nba = np.linalg.norm(ba)
# Executes this step as part of the end to end video pose analytics workflow.
nbc = np.linalg.norm(bc)
# Applies a guard condition to handle missing data and enforce expected preconditions.
if nba < 1e-6 or nbc < 1e-6:
    # Returns the computed value to the caller for downstream use.
    return np.nan
# Executes this step as part of the end to end video pose analytics workflow.
cosang = float(np.clip(np.dot(ba, bc) / (nba * nbc), -1.0, 1.0))
# Returns the computed value to the caller for downstream use.
return float(np.degrees(np.arccos(cosang)))

        # Defines _valid_xy to encapsulate reusable logic for the pipeline.
        def _valid_xy(p):












# Returns the computed value to the caller for downstream use.
return (p is not None) and np.isfinite(p).all() and (p[0] > 0) and (p[1] > 0)
        # Defines ready_score_from_kpts to encapsulate reusable logic for the pipeline.
        def ready_score_from_kpts(kpts_xy):
"""
ReadyScore in [0,100], HIGHER is better.
100 = excellent ready posture; 0 = poor readiness.
"""
# Executes this step as part of the end to end video pose analytics workflow.
k = np.asarray(kpts_xy, dtype=float)
# Applies a guard condition to handle missing data and enforce expected preconditions.
if k.shape != (17, 2):
    # Returns the computed value to the caller for downstream use.
    return np.nan


# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
L_EL, R_EL = 7, 8
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
L_WR, R_WR = 9, 10
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
L_HI, R_HI = 11, 12
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
L_KN, R_KN = 13, 14
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
L_AN, R_AN = 15, 16

# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
req = [L_SH, R_SH, L_HI, R_HI, L_KN, R_KN, L_AN, R_AN]
# Applies a guard condition to handle missing data and enforce expected preconditions.
if not all(_valid_xy(k[i]) for i in req):
    # Returns the computed value to the caller for downstream use.
    return np.nan

# Executes this step as part of the end to end video pose analytics workflow.
sh_mid = (k[L_SH] + k[R_SH]) / 2.0
# Executes this step as part of the end to end video pose analytics workflow.
hi_mid = (k[L_HI] + k[R_HI]) / 2.0
# Executes this step as part of the end to end video pose analytics workflow.
shoulder_w = _dist(k[L_SH], k[R_SH])
# Executes this step as part of the end to end video pose analytics workflow.
_ = max(shoulder_w, _dist(sh_mid, hi_mid), 1.0) # scale placeholder (kept for clarity)
# Executes this step as part of the end to end video pose analytics workflow.
lk = _angle(k[L_HI], k[L_KN], k[L_AN])
# Executes this step as part of the end to end video pose analytics workflow.
rk = _angle(k[R_HI], k[R_KN], k[R_AN])
# Executes this step as part of the end to end video pose analytics workflow.
knee_ang = np.nanmean([lk, rk])
# Applies a guard condition to handle missing data and enforce expected preconditions.
if not np.isfinite(knee_ang):
    # Returns the computed value to the caller for downstream use.
    return np.nan

# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
knee_target = 145.0
# Executes this step as part of the end to end video pose analytics workflow.
knee_pen = abs(knee_ang - knee_target) / 45.0

# Executes this step as part of the end to end video pose analytics workflow.
stance = _dist(k[L_AN], k[R_AN]) / max(shoulder_w, 1.0)
# Applies a guard condition to handle missing data and enforce expected preconditions.
if stance < 1.1:
    # Executes this step as part of the end to end video pose analytics workflow.
    stance_pen = (1.1 - stance) / 0.6
# Executes this step as part of the end to end video pose analytics workflow.
elif stance > 2.0:
    # Executes this step as part of the end to end video pose analytics workflow.
    stance_pen = (stance - 2.0) / 1.0
# Executes this step as part of the end to end video pose analytics workflow.
else:









    # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
    stance_pen = 0.0

# Executes this step as part of the end to end video pose analytics workflow.
torso_vec = sh_mid - hi_mid
# Executes this step as part of the end to end video pose analytics workflow.
n = np.linalg.norm(torso_vec)
# Applies a guard condition to handle missing data and enforce expected preconditions.
if n < 1e-6:
    # Returns the computed value to the caller for downstream use.
    return np.nan
# Executes this step as part of the end to end video pose analytics workflow.
v = torso_vec / n
# Executes this step as part of the end to end video pose analytics workflow.
vert = np.array([0.0, -1.0])
# Executes this step as part of the end to end video pose analytics workflow.
lean_deg = float(np.degrees(np.arccos(np.clip(np.dot(v, vert), -1.0, 1.0))))
# Applies a guard condition to handle missing data and enforce expected preconditions.
if lean_deg < 10:
    # Executes this step as part of the end to end video pose analytics workflow.
    lean_pen = (10 - lean_deg) / 15.0
# Executes this step as part of the end to end video pose analytics workflow.
elif lean_deg > 25:
    # Executes this step as part of the end to end video pose analytics workflow.
    lean_pen = (lean_deg - 25) / 25.0
# Executes this step as part of the end to end video pose analytics workflow.
else:
    # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
    lean_pen = 0.0


# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
hand_pen = 0.5
# Executes this step as part of the end to end video pose analytics workflow.
wrists_ok = _valid_xy(k[L_WR]) and _valid_xy(k[R_WR])
# Applies a guard condition to handle missing data and enforce expected preconditions.
if wrists_ok:
    # Executes this step as part of the end to end video pose analytics workflow.
    wr_mid = (k[L_WR] + k[R_WR]) / 2.0
    # Executes this step as part of the end to end video pose analytics workflow.
    denom = max((hi_mid[1] - sh_mid[1]), 1.0)
    # Executes this step as part of the end to end video pose analytics workflow.
    r = float((wr_mid[1] - sh_mid[1]) / denom)
    # Applies a guard condition to handle missing data and enforce expected preconditions.
    if r < 0.2:
        # Executes this step as part of the end to end video pose analytics workflow.
        vpen = (0.2 - r) / 0.2
    # Executes this step as part of the end to end video pose analytics workflow.
    elif r > 0.85:
        # Executes this step as part of the end to end video pose analytics workflow.
        vpen = (r - 0.85) / 0.35
    # Executes this step as part of the end to end video pose analytics workflow.
    else:
        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        vpen = 0.0
    # Executes this step as part of the end to end video pose analytics workflow.
    hpen = abs(float(wr_mid[0] - sh_mid[0])) / max(shoulder_w, 1.0)
    # Executes this step as part of the end to end video pose analytics workflow.
    hpen = max(0.0, (hpen - 0.7) / 0.8)
    # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
    hand_pen = 0.65 * vpen + 0.35 * hpen

# Executes this step as part of the end to end video pose analytics workflow.
raw_bad = (1.20 * knee_pen) + (0.90 * stance_pen) + (0.70 * lean_pen) + (0.
            # Executes this step as part of the end to end video pose analytics workflow.
            80 * hand_pen)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
bad_score = 100.0 * (raw_bad / 4.0)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
readiness = 100.0 - float(np.clip(bad_score, 0.0, 100.0))
# Returns the computed value to the caller for downstream use.
return float(np.clip(readiness, 0.0, 100.0))

In [ ]:

# Applies a guard condition to handle missing data and enforce expected preconditions.
if "MODEL_WEIGHTS" not in globals():
    # Stores a configuration constant to standardize inputs and outputs across runs.
    MODEL_WEIGHTS = "yolov8n-pose.pt"
# Applies a guard condition to handle missing data and enforce expected preconditions.
if "CONF" not in globals():










    # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
    CONF = 0.25

# Initializes the pose estimation model using the configured weights.
model = YOLO(MODEL_WEIGHTS)
# Logs key intermediate outputs to support debugging and auditability.
print("Loaded model:", MODEL_WEIGHTS, "| CONF:", CONF)

# Executes this step as part of the end to end video pose analytics workflow.
Loaded model: yolov8n-pose.pt | CONF: 0.25


In [ ]:

        # Imports required packages to support reproducible data processing and model inference.
        import re
        # Imports required packages to support reproducible data processing and model inference.
        import numpy as np
        # Imports required packages to support reproducible data processing and model inference.
        import cv2

        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        SELECT_T_SEC = 1.0

        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        TARGET_DESC = {
# Executes this step as part of the end to end video pose analytics workflow.
"AM1": "pink shirt",
# Executes this step as part of the end to end video pose analytics workflow.
"AM2": "pink shirt",
# Executes this step as part of the end to end video pose analytics workflow.
"AM3": "pink shirt",
# Executes this step as part of the end to end video pose analytics workflow.
"PR1": "blue shirt and white pants",
# Executes this step as part of the end to end video pose analytics workflow.
"PR2": "blue shirt and white pants",
# Executes this step as part of the end to end video pose analytics workflow.
"PR3": "blue shirt and white pants",

        # Executes this step as part of the end to end video pose analytics workflow.
        }

        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        HSV_RANGES = {
# Executes this step as part of the end to end video pose analytics workflow.
"red": [((0, 80, 60), (10, 255, 255)), ((170, 80, 60), (179, 255, 255))],
# Executes this step as part of the end to end video pose analytics workflow.
"orange":[((10, 80, 60), (22, 255, 255))],
# Executes this step as part of the end to end video pose analytics workflow.
"yellow":[((22, 80, 60), (35, 255, 255))],
# Executes this step as part of the end to end video pose analytics workflow.
"green": [((35, 60, 50), (85, 255, 255))],
# Executes this step as part of the end to end video pose analytics workflow.
"blue": [((90, 60, 50), (130, 255, 255))],
# Executes this step as part of the end to end video pose analytics workflow.
"purple":[((130, 50, 50), (165, 255, 255))],
# Executes this step as part of the end to end video pose analytics workflow.
"pink": [((145, 50, 60), (170, 255, 255))],
# Executes this step as part of the end to end video pose analytics workflow.
"white": [((0, 0, 200), (179, 60, 255))],
# Executes this step as part of the end to end video pose analytics workflow.
"black": [((0, 0, 0), (179, 255, 50))],
# Executes this step as part of the end to end video pose analytics workflow.
"gray": [((0, 0, 50), (179, 40, 200))],
# Executes this step as part of the end to end video pose analytics workflow.
"navy": [((100, 80, 20), (135, 255, 120))], # rough
        # Executes this step as part of the end to end video pose analytics workflow.
        }












        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        REGION_SYNONYMS = {
# Executes this step as part of the end to end video pose analytics workflow.
"hat": "hat",
# Executes this step as part of the end to end video pose analytics workflow.
"cap": "hat",
# Executes this step as part of the end to end video pose analytics workflow.
"shirt": "shirt",
# Executes this step as part of the end to end video pose analytics workflow.
"top": "shirt",
# Executes this step as part of the end to end video pose analytics workflow.
"tee": "shirt",
# Executes this step as part of the end to end video pose analytics workflow.
"tshirt": "shirt",
# Executes this step as part of the end to end video pose analytics workflow.
"shorts": "shorts",
# Executes this step as part of the end to end video pose analytics workflow.
"pants": "shorts", # treat as lower body region
# Executes this step as part of the end to end video pose analytics workflow.
"trousers": "shorts",
        # Executes this step as part of the end to end video pose analytics workflow.
        }

        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        REGION_WEIGHTS = {
# Executes this step as part of the end to end video pose analytics workflow.
"hat": 0.55,

# Executes this step as part of the end to end video pose analytics workflow.
"shirt": 0.45,
# Executes this step as part of the end to end video pose analytics workflow.
"shorts": 0.35,
# Executes this step as part of the end to end video pose analytics workflow.
None: 0.40, # if user says just "blue" with no region
        # Executes this step as part of the end to end video pose analytics workflow.
        }

        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        MIN_PER_CONSTRAINT = {
# Executes this step as part of the end to end video pose analytics workflow.
"hat": 0.015,
# Executes this step as part of the end to end video pose analytics workflow.
"shirt": 0.020,
# Executes this step as part of the end to end video pose analytics workflow.
"shorts": 0.020,
# Executes this step as part of the end to end video pose analytics workflow.
None: 0.020,
        # Executes this step as part of the end to end video pose analytics workflow.
        }

        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        MIN_TOTAL_SCORE = 0.03


        # Defines _extract_frame_at_time to encapsulate reusable logic for the pipeline.
        def _extract_frame_at_time(video_path, t_sec=1.0):
# Opens the input video stream to read metadata and frames.
cap = cv2.VideoCapture(video_path)
# Applies a guard condition to handle missing data and enforce expected preconditions.
if not cap.isOpened():
    # Executes this step as part of the end to end video pose analytics workflow.
    raise RuntimeError(f"Could not open video: {video_path}")
# Executes this step as part of the end to end video pose analytics workflow.
cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, float(t_sec)) * 1000.0)
# Executes this step as part of the end to end video pose analytics workflow.
ok, frame_bgr = cap.read()
# Executes this step as part of the end to end video pose analytics workflow.
cap.release()

# Applies a guard condition to handle missing data and enforce expected preconditions.
if not ok or frame_bgr is None:
    # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
    raise RuntimeError(f"Could not read frame at t={t_sec}s from {video_path}")
# Returns the computed value to the caller for downstream use.
return frame_bgr
        # Defines _predict_person_boxes_on_frame to encapsulate reusable logic for the pipeline.
        def _predict_person_boxes_on_frame(model, frame_bgr, conf=0.25):
# Executes this step as part of the end to end video pose analytics workflow.
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)










# Executes this step as part of the end to end video pose analytics workflow.
r = model.predict(frame_rgb, conf=conf, verbose=False)[0]
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
if r.boxes is None or len(r.boxes) == 0:
    # Returns the computed value to the caller for downstream use.
    return np.zeros((0, 4), dtype=int), frame_rgb
# Executes this step as part of the end to end video pose analytics workflow.
boxes = r.boxes.xyxy.cpu().numpy().astype(int)
# Returns the computed value to the caller for downstream use.
return boxes, frame_rgb

        # Defines region_crop_from_bbox to encapsulate reusable logic for the pipeline.
        def region_crop_from_bbox(img_rgb, bbox_xyxy, region=None):
"""
region heuristic within person bbox:
hat:  top 25%
shirt: middle 45%
shorts: bottom 45%
None: entire bbox
"""
# Executes this step as part of the end to end video pose analytics workflow.
x1, y1, x2, y2 = map(int, bbox_xyxy)
# Executes this step as part of the end to end video pose analytics workflow.
x1 = max(0, x1); y1 = max(0, y1)

# Executes this step as part of the end to end video pose analytics workflow.
x2 = min(img_rgb.shape[1]-1, x2); y2 = min(img_rgb.shape[0]-1, y2)
# Applies a guard condition to handle missing data and enforce expected preconditions.
if x2 <= x1 or y2 <= y1:
    # Returns the computed value to the caller for downstream use.
    return None

# Executes this step as part of the end to end video pose analytics workflow.
h = y2 - y1
# Stores a configuration constant to standardize inputs and outputs across runs.
if region == "hat":
    # Executes this step as part of the end to end video pose analytics workflow.
    yy1, yy2 = y1, y1 + int(0.25 * h)
# Stores a configuration constant to standardize inputs and outputs across runs.
elif region == "shirt":
    # Executes this step as part of the end to end video pose analytics workflow.
    yy1, yy2 = y1 + int(0.20 * h), y1 + int(0.65 * h)
# Stores a configuration constant to standardize inputs and outputs across runs.
elif region == "shorts":
    # Executes this step as part of the end to end video pose analytics workflow.
    yy1, yy2 = y1 + int(0.55 * h), y2
# Executes this step as part of the end to end video pose analytics workflow.
else:
    # Executes this step as part of the end to end video pose analytics workflow.
    yy1, yy2 = y1, y2

# Executes this step as part of the end to end video pose analytics workflow.
yy1 = max(y1, yy1)
# Executes this step as part of the end to end video pose analytics workflow.
yy2 = min(y2, yy2)
# Applies a guard condition to handle missing data and enforce expected preconditions.
if yy2 <= yy1:
    # Returns the computed value to the caller for downstream use.
    return None

# Returns the computed value to the caller for downstream use.
return img_rgb[yy1:yy2, x1:x2].copy()

        # Defines color_fraction_hsv to encapsulate reusable logic for the pipeline.
        def color_fraction_hsv(img_rgb_crop, color_name):
"""Fraction of pixels matching the target color (HSV threshold)."""
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
if img_rgb_crop is None or img_rgb_crop.size == 0:

    # Returns the computed value to the caller for downstream use.
    return 0.0
# Executes this step as part of the end to end video pose analytics workflow.
hsv = cv2.cvtColor(img_rgb_crop, cv2.COLOR_RGB2HSV)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
masks = []
# Iterates through streamed results to accumulate frame level metrics.
for (lo, hi) in HSV_RANGES[color_name]:
    # Executes this step as part of the end to end video pose analytics workflow.
    lo = np.array(lo, dtype=np.uint8)
    # Executes this step as part of the end to end video pose analytics workflow.
    hi = np.array(hi, dtype=np.uint8)
    # Executes this step as part of the end to end video pose analytics workflow.
    masks.append(cv2.inRange(hsv, lo, hi))










# Executes this step as part of the end to end video pose analytics workflow.
mask = masks[0]
# Iterates through streamed results to accumulate frame level metrics.
for m in masks[1:]:
    # Executes this step as part of the end to end video pose analytics workflow.
    mask = cv2.bitwise_or(mask, m)
# Returns the computed value to the caller for downstream use.
return float(mask.mean() / 255.0)

        # Defines parse_multi_desc to encapsulate reusable logic for the pipeline.
        def parse_multi_desc(desc: str):
"""
Parse multiple constraints from:
"blue hat and red shirt" -> [("blue","hat"),("red","shirt")]
"white shirt, black shorts" -> ...
"blue" -> [("blue", None)]
"""
# Executes this step as part of the end to end video pose analytics workflow.
d = desc.lower().strip()
# Executes this step as part of the end to end video pose analytics workflow.
chunks = re.split(r"\s*(?:and|&|\+|,|;)\s*", d)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
constraints = []
# Iterates through streamed results to accumulate frame level metrics.
for ch in chunks:

    # Executes this step as part of the end to end video pose analytics workflow.
    toks = re.split(r"\s+", ch.strip())
    # Executes this step as part of the end to end video pose analytics workflow.
    color = None
    # Executes this step as part of the end to end video pose analytics workflow.
    region = None
    # Iterates through streamed results to accumulate frame level metrics.
    for t in toks:
        # Applies a guard condition to handle missing data and enforce expected preconditions.
        if t in HSV_RANGES:
        # Executes this step as part of the end to end video pose analytics workflow.
        color = t
        # Applies a guard condition to handle missing data and enforce expected preconditions.
        if t in REGION_SYNONYMS:
        # Executes this step as part of the end to end video pose analytics workflow.
        region = REGION_SYNONYMS[t]
    # Applies a guard condition to handle missing data and enforce expected preconditions.
    if color is not None:
        # Executes this step as part of the end to end video pose analytics workflow.
        constraints.append((color, region))

# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
if len(constraints) == 0:
    # Executes this step as part of the end to end video pose analytics workflow.
    raise ValueError(
        # Executes this step as part of the end to end video pose analytics workflow.
        f"Could not parse any (color, region) from '{desc}'. "
        # Executes this step as part of the end to end video pose analytics workflow.
        f"Known colors: {list(HSV_RANGES.keys())}. Regions: {sorted(set(REGION_SYNONYMS.keys()))}"
    # Executes this step as part of the end to end video pose analytics workflow.
    )
# Returns the computed value to the caller for downstream use.
return constraints

        # Defines score_person_for_constraints to encapsulate reusable logic for the pipeline.
        def score_person_for_constraints(frame_rgb, bbox_xyxy, constraints):
"""

Returns:
total_score (weighted sum, normalized),
per_constraint list of dicts.
"""
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
per = []
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
total = 0.0
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
weight_sum = 0.0

# Iterates through streamed results to accumulate frame level metrics.
for (color, region) in constraints:










    # Executes this step as part of the end to end video pose analytics workflow.
    crop = region_crop_from_bbox(frame_rgb, bbox_xyxy, region=region)
    # Executes this step as part of the end to end video pose analytics workflow.
    frac = color_fraction_hsv(crop, color)

    # Executes this step as part of the end to end video pose analytics workflow.
    w = float(REGION_WEIGHTS.get(region, REGION_WEIGHTS.get(None, 0.4)))
    # Executes this step as part of the end to end video pose analytics workflow.
    weight_sum += w
    # Executes this step as part of the end to end video pose analytics workflow.
    total += w * frac

    # Executes this step as part of the end to end video pose analytics workflow.
    per.append({
        # Executes this step as part of the end to end video pose analytics workflow.
        "color": color,
        # Executes this step as part of the end to end video pose analytics workflow.
        "region": region,
        # Executes this step as part of the end to end video pose analytics workflow.
        "frac": float(frac),
        # Executes this step as part of the end to end video pose analytics workflow.
        "weight": float(w),
        # Executes this step as part of the end to end video pose analytics workflow.
        "min_required": float(MIN_PER_CONSTRAINT.get(region, 0.02)),
    # Executes this step as part of the end to end video pose analytics workflow.
    })

# Applies a guard condition to handle missing data and enforce expected preconditions.
if weight_sum > 1e-9:

    # Executes this step as part of the end to end video pose analytics workflow.
    total = total / weight_sum

# Returns the computed value to the caller for downstream use.
return float(total), per

        # Defines select_target_bbox_by_multi_desc to encapsulate reusable logic for the pipeline.
        def select_target_bbox_by_multi_desc(video_path, model, desc, conf=0.25, t_sec=1.0):
# Executes this step as part of the end to end video pose analytics workflow.
constraints = parse_multi_desc(desc)
# Executes this step as part of the end to end video pose analytics workflow.
frame_bgr = _extract_frame_at_time(video_path, t_sec=t_sec)
# Executes this step as part of the end to end video pose analytics workflow.
boxes, frame_rgb = _predict_person_boxes_on_frame(model, frame_bgr, conf=conf)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
if boxes.shape[0] == 0:
    # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
    raise RuntimeError(f"No persons detected for {video_path} at t={t_sec}s")
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
best = {"idx": None, "score": -1.0, "per": None}
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
all_debug = []
# Iterates through streamed results to accumulate frame level metrics.
for i, b in enumerate(boxes):

    # Executes this step as part of the end to end video pose analytics workflow.
    total, per = score_person_for_constraints(frame_rgb, b, constraints)

    # Executes this step as part of the end to end video pose analytics workflow.
    ok = True
    # Iterates through streamed results to accumulate frame level metrics.
    for p in per:
        # Applies a guard condition to handle missing data and enforce expected preconditions.
        if p["frac"] < p["min_required"]:
        # Executes this step as part of the end to end video pose analytics workflow.
        ok = False
        # Executes this step as part of the end to end video pose analytics workflow.
        break

    # Executes this step as part of the end to end video pose analytics workflow.
    all_debug.append({"i": i, "total": total, "ok": ok, "per": per})

    # Applies a guard condition to handle missing data and enforce expected preconditions.
    if ok and total > best["score"]:
        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        best = {"idx": int(i), "score": float(total), "per": per}











# Applies a guard condition to handle missing data and enforce expected preconditions.
if best["idx"] is None or best["score"] < MIN_TOTAL_SCORE:
    # Executes this step as part of the end to end video pose analytics workflow.
    top = sorted(all_debug, key=lambda x: x["total"], reverse=True)[:3]
    # Executes this step as part of the end to end video pose analytics workflow.
    msg = (
        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        f"Could not confidently match '{desc}' at t={t_sec}s.\n"
        # Executes this step as part of the end to end video pose analytics workflow.
        f"Best passing score: {best['score']:.4f} (min total {MIN_TOTAL_SCORE}).\n"
        # Executes this step as part of the end to end video pose analytics workflow.
        "Top candidates (even if failing per-constraint mins):\n"
    # Executes this step as part of the end to end video pose analytics workflow.
    )
    # Iterates through streamed results to accumulate frame level metrics.
    for cand in top:
        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        msg += f" - idx {cand['i']}: total={cand['total']:.4f}, ok={cand['ok']}, per={cand['per']}\n"
    # Executes this step as part of the end to end video pose analytics workflow.
    msg += (
        # Executes this step as part of the end to end video pose analytics workflow.
        "\nTips:\n"
        # Executes this step as part of the end to end video pose analytics workflow.
        "- Try a different SELECT_T_SEC (e.g., 2.0–4.0)\n"
        # Executes this step as part of the end to end video pose analytics workflow.
        "- Use larger/clearer region like 'red shirt' instead of 'blue hat'\n"
        # Executes this step as part of the end to end video pose analytics workflow.
        "- Lower MIN_PER_CONSTRAINT or MIN_TOTAL_SCORE slightly if lighting is tough\n"
    # Executes this step as part of the end to end video pose analytics workflow.
    )
    # Executes this step as part of the end to end video pose analytics workflow.
    raise RuntimeError(msg)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
debug = {
    # Executes this step as part of the end to end video pose analytics workflow.
    "desc": desc,
    # Executes this step as part of the end to end video pose analytics workflow.
    "constraints": constraints,
    # Executes this step as part of the end to end video pose analytics workflow.
    "best_index": best["idx"],
    # Executes this step as part of the end to end video pose analytics workflow.
    "best_score": best["score"],
    # Executes this step as part of the end to end video pose analytics workflow.
    "best_per": best["per"],
    # Executes this step as part of the end to end video pose analytics workflow.
    "t_sec": float(t_sec),
    # Executes this step as part of the end to end video pose analytics workflow.
    "num_people": int(len(boxes)),
    # Executes this step as part of the end to end video pose analytics workflow.
    "all_debug": all_debug, # can comment out if too verbose
# Executes this step as part of the end to end video pose analytics workflow.
}
# Returns the computed value to the caller for downstream use.
return boxes[best["idx"]].astype(float), debug










        # Executes this step as part of the end to end video pose analytics workflow.
        PRO_TARGET_BBOX_2, dbg_pr2 = select_target_bbox_by_multi_desc(PRO_CLIP_2, model, TARGET_DESC["PR2"], conf=CONF, t_sec=SELECT_T_SEC)
        # Executes this step as part of the end to end video pose analytics workflow.
        PRO_TARGET_BBOX_3, dbg_pr3 = select_target_bbox_by_multi_desc(PRO_CLIP_3, model, TARGET_DESC["PR3"], conf=CONF, t_sec=SELECT_T_SEC)
        # Logs key intermediate outputs to support debugging and auditability.
        print("Selected target bboxes (xyxy):")
        # Logs key intermediate outputs to support debugging and auditability.
        print(" AM1:", AM_TARGET_BBOX)
        # Logs key intermediate outputs to support debugging and auditability.
        print(" AM2:", AM_TARGET_BBOX_2)
        # Logs key intermediate outputs to support debugging and auditability.
        print(" AM3:", AM_TARGET_BBOX_3)
        # Logs key intermediate outputs to support debugging and auditability.
        print(" PR1:", PRO_TARGET_BBOX)
        # Logs key intermediate outputs to support debugging and auditability.
        print(" PR2:", PRO_TARGET_BBOX_2)
        # Logs key intermediate outputs to support debugging and auditability.
        print(" PR3:", PRO_TARGET_BBOX_3)

        # Logs key intermediate outputs to support debugging and auditability.
        print("\nBest-match debug:")
        # Logs key intermediate outputs to support debugging and auditability.
        print(" AM1:", {"best_index": dbg_am1["best_index"], "best_score": dbg_am1["best_score"], "best_per": dbg_am1["best_per"]})
        # Logs key intermediate outputs to support debugging and auditability.
        print(" PR1:", {"best_index": dbg_pr1["best_index"], "best_score": dbg_pr1["best_score"], "best_per": dbg_pr1["best_per"]})
        # Executes this step as part of the end to end video pose analytics workflow.
        Selected target bboxes (xyxy):
            # Executes this step as part of the end to end video pose analytics workflow.
            AM1: [     717       28       776      163]
            # Executes this step as part of the end to end video pose analytics workflow.
            AM2: [     782       11       854      138]
            # Executes this step as part of the end to end video pose analytics workflow.
            AM3: [     774        3       835      142]
            # Executes this step as part of the end to end video pose analytics workflow.
            PR1: [     463      177       539      348]
            # Executes this step as part of the end to end video pose analytics workflow.
            PR2: [    1059      352      1176      716]
            # Executes this step as part of the end to end video pose analytics workflow.
            PR3: [     646      162       721      295]
        # Executes this step as part of the end to end video pose analytics workflow.
        Best-match debug:
            # Executes this step as part of the end to end video pose analytics workflow.
            AM1: {'best_index': 1, 'best_score': 0.24689265536723165, 'best_per':
        # Executes this step as part of the end to end video pose analytics workflow.
        [{'color': 'pink', 'region': 'shirt', 'frac': 0.24689265536723165, 'weight':
        # Executes this step as part of the end to end video pose analytics workflow.
        0.45, 'min_required': 0.02}]}
            # Executes this step as part of the end to end video pose analytics workflow.
            PR1: {'best_index': 2, 'best_score': 0.20325316131237184, 'best_per':
        # Executes this step as part of the end to end video pose analytics workflow.
        [{'color': 'blue', 'region': 'shirt', 'frac': 0.31681476418318527, 'weight':
        # Executes this step as part of the end to end video pose analytics workflow.
        0.45, 'min_required': 0.02}, {'color': 'white', 'region': 'shorts', 'frac':
        # Executes this step as part of the end to end video pose analytics workflow.
        0.05724538619275461, 'weight': 0.35, 'min_required': 0.02}]}

        # Executes an external command to perform system level media or data operations.
        run_cmd([“gsutil”,“-m”,“cp”,“-r”,“01_yolo_pose_baseline.ipynb”,“runs/”,“gs://…/”])

        # Imports required packages to support reproducible data processing and model inference.
        import matplotlib.pyplot as plt
        # Imports required packages to support reproducible data processing and model inference.
        import numpy as np
        # Imports required packages to support reproducible data processing and model inference.
        import cv2

        # Defines draw_all_players_and_target to encapsulate reusable logic for the pipeline.
        def draw_all_players_and_target(video_path, model, target_bbox, t_sec=1.0, conf=0.25, title=""):
# Executes this step as part of the end to end video pose analytics workflow.
frame_bgr = _extract_frame_at_time(video_path, t_sec=t_sec)









# Executes this step as part of the end to end video pose analytics workflow.
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


# Iterates through streamed results to accumulate frame level metrics.
for i, b in enumerate(boxes):
    # Executes this step as part of the end to end video pose analytics workflow.
    x1, y1, x2, y2 = map(int, b)
    # Executes this step as part of the end to end video pose analytics workflow.
    cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), (255, 255, 0), 2) # yellow
    # Executes this step as part of the end to end video pose analytics workflow.
    cv2.putText(frame_rgb, f"{i}", (x1, max(0, y1 - 6)),
            # Executes this step as part of the end to end video pose analytics workflow.
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2, cv2.
            # Executes this step as part of the end to end video pose analytics workflow.
            LINE_AA)
# Applies a guard condition to handle missing data and enforce expected preconditions.
if target_bbox is not None:

    # Executes this step as part of the end to end video pose analytics workflow.
    x1, y1, x2, y2 = map(int, target_bbox)
    # Executes this step as part of the end to end video pose analytics workflow.
    cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), (0, 255, 0), 4) # green
    # Executes this step as part of the end to end video pose analytics workflow.
    cv2.putText(frame_rgb, "TARGET", (x1, max(0, y1 - 28)),
            # Executes this step as part of the end to end video pose analytics workflow.
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2, cv2.LINE_AA)

# Executes this step as part of the end to end video pose analytics workflow.
plt.figure(figsize=(14, 7))
# Executes this step as part of the end to end video pose analytics workflow.
plt.imshow(frame_rgb)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
plt.title(title + f" (t={t_sec:.1f}s, detected={len(boxes)})")
# Executes this step as part of the end to end video pose analytics workflow.
plt.axis("off")
# Executes this step as part of the end to end video pose analytics workflow.
plt.show()

        # Executes this step as part of the end to end video pose analytics workflow.
        draw_all_players_and_target(AM_CLIP, model, AM_TARGET_BBOX, t_sec=SELECT_T_SEC, conf=CONF, title=f"AM1 — {TARGET_DESC['AM1']}")
        # Executes this step as part of the end to end video pose analytics workflow.
        draw_all_players_and_target(AM_CLIP_2, model, AM_TARGET_BBOX_2, t_sec=SELECT_T_SEC, conf=CONF, title=f"AM2 — {TARGET_DESC['AM2']}")
        # Executes this step as part of the end to end video pose analytics workflow.
        draw_all_players_and_target(AM_CLIP_3, model, AM_TARGET_BBOX_3, t_sec=SELECT_T_SEC, conf=CONF, title=f"AM3 — {TARGET_DESC['AM3']}")
        # Executes this step as part of the end to end video pose analytics workflow.
        draw_all_players_and_target(PRO_CLIP, model, PRO_TARGET_BBOX, t_sec=SELECT_T_SEC, conf=CONF, title=f"PR1 — {TARGET_DESC['PR1']}")
        # Executes this step as part of the end to end video pose analytics workflow.
        draw_all_players_and_target(PRO_CLIP_2, model, PRO_TARGET_BBOX_2, t_sec=SELECT_T_SEC, conf=CONF, title=f"PR2 — {TARGET_DESC['PR2']}")
        # Executes this step as part of the end to end video pose analytics workflow.
        draw_all_players_and_target(PRO_CLIP_3, model, PRO_TARGET_BBOX_3, t_sec=SELECT_T_SEC, conf=CONF, title=f"PR3 — {TARGET_DESC['PR3']}")

In [ ]:
        # Defines _iou_xyxy to encapsulate reusable logic for the pipeline.
        def _iou_xyxy(a, b):
# Executes this step as part of the end to end video pose analytics workflow.
ax1, ay1, ax2, ay2 = map(float, a)










# Executes this step as part of the end to end video pose analytics workflow.
bx1, by1, bx2, by2 = map(float, b)
# Executes this step as part of the end to end video pose analytics workflow.
ix1, iy1 = max(ax1, bx1), max(ay1, by1)
# Executes this step as part of the end to end video pose analytics workflow.
ix2, iy2 = min(ax2, bx2), min(ay2, by2)
# Executes this step as part of the end to end video pose analytics workflow.
iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
# Executes this step as part of the end to end video pose analytics workflow.
inter = iw * ih
# Executes this step as part of the end to end video pose analytics workflow.
area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
# Executes this step as part of the end to end video pose analytics workflow.
area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
# Executes this step as part of the end to end video pose analytics workflow.
union = area_a + area_b - inter
# Returns the computed value to the caller for downstream use.
return inter / union if union > 1e-9 else 0.0

        # Defines summarize_video_ready_score_locked to encapsulate reusable logic for the pipeline.
        def summarize_video_ready_score_locked(
# Executes this step as part of the end to end video pose analytics workflow.
video_path,
# Executes this step as part of the end to end video pose analytics workflow.
model,
# Executes this step as part of the end to end video pose analytics workflow.
init_bbox_xyxy,
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
conf=0.25,
# Stores a configuration constant to standardize inputs and outputs across runs.
tracker_cfg="bytetrack.yaml",

# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
init_search_frames=45,
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
reacquire_iou_thresh=0.25,
        # Executes this step as part of the end to end video pose analytics workflow.
        ):
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
scores = []
# Executes this step as part of the end to end video pose analytics workflow.
target_id = None
# Executes this step as part of the end to end video pose analytics workflow.
last_bbox = None
# Executes this step as part of the end to end video pose analytics workflow.
init_bbox = np.array(init_bbox_xyxy, dtype=float)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
frame_idx = 0

# Runs multi object tracking with pose keypoints to generate frame level observations.
for r in model.track(
# Executes this step as part of the end to end video pose analytics workflow.
source=video_path,
# Executes this step as part of the end to end video pose analytics workflow.
conf=conf,
# Executes this step as part of the end to end video pose analytics workflow.
stream=True,
# Executes this step as part of the end to end video pose analytics workflow.
persist=True,
# Executes this step as part of the end to end video pose analytics workflow.
tracker=tracker_cfg,
# Executes this step as part of the end to end video pose analytics workflow.
verbose=False,
# Executes this step as part of the end to end video pose analytics workflow.
):
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
frame_idx += 1

# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
if r.boxes is None or len(r.boxes) == 0 or r.keypoints is None:
    # Executes this step as part of the end to end video pose analytics workflow.
    continue

# Applies a guard condition to handle missing data and enforce expected preconditions.
if getattr(r.boxes, "id", None) is None:

    # Executes this step as part of the end to end video pose analytics workflow.
    boxes = r.boxes.xyxy.cpu().numpy().astype(float)
    # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
    if boxes.shape[0] == 0:
        # Executes this step as part of the end to end video pose analytics workflow.
        continue
    # Executes this step as part of the end to end video pose analytics workflow.
    ious = np.array([_iou_xyxy(init_bbox, b) for b in boxes], dtype=float)
    # Executes this step as part of the end to end video pose analytics workflow.
    i = int(np.argmax(ious))
    # Applies a guard condition to handle missing data and enforce expected preconditions.
    if ious[i] < 0.05:









        # Executes this step as part of the end to end video pose analytics workflow.
        continue
    # Executes this step as part of the end to end video pose analytics workflow.
    kpts = r.keypoints.xy[i].cpu().numpy()
    # Executes this step as part of the end to end video pose analytics workflow.
    s = ready_score_from_kpts(kpts)
    # Applies a guard condition to handle missing data and enforce expected preconditions.
    if np.isfinite(s):
        # Executes this step as part of the end to end video pose analytics workflow.
        scores.append(float(s))
    # Executes this step as part of the end to end video pose analytics workflow.
    continue

# Executes this step as part of the end to end video pose analytics workflow.
boxes = r.boxes.xyxy.cpu().numpy().astype(float)
# Executes this step as part of the end to end video pose analytics workflow.
ids = r.boxes.id.cpu().numpy().astype(int)

# Applies a guard condition to handle missing data and enforce expected preconditions.
if target_id is None and frame_idx <= init_search_frames:
    # Executes this step as part of the end to end video pose analytics workflow.
    ious = np.array([_iou_xyxy(init_bbox, b) for b in boxes], dtype=float)
    # Executes this step as part of the end to end video pose analytics workflow.
    best = int(np.argmax(ious))
    # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
    if ious[best] >= 0.10:
        # Executes this step as part of the end to end video pose analytics workflow.
        target_id = int(ids[best])
        # Executes this step as part of the end to end video pose analytics workflow.
        last_bbox = boxes[best].copy()

# Executes this step as part of the end to end video pose analytics workflow.
idxs = np.where(ids == target_id)[0] if target_id is not None else np.
            # Executes this step as part of the end to end video pose analytics workflow.
            array([], dtype=int)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
if target_id is not None and len(idxs) == 0 and last_bbox is not None:
    # Executes this step as part of the end to end video pose analytics workflow.
    ious = np.array([_iou_xyxy(last_bbox, b) for b in boxes], dtype=float)
    # Executes this step as part of the end to end video pose analytics workflow.
    best = int(np.argmax(ious))
    # Applies a guard condition to handle missing data and enforce expected preconditions.
    if ious[best] >= reacquire_iou_thresh:
        # Executes this step as part of the end to end video pose analytics workflow.
        target_id = int(ids[best])
        # Executes this step as part of the end to end video pose analytics workflow.
        idxs = np.array([best], dtype=int)
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
if len(idxs) == 0:
    # Executes this step as part of the end to end video pose analytics workflow.
    continue

# Executes this step as part of the end to end video pose analytics workflow.
i = int(idxs[0])

# Executes this step as part of the end to end video pose analytics workflow.
last_bbox = boxes[i].copy()
# Executes this step as part of the end to end video pose analytics workflow.
kpts = r.keypoints.xy[i].cpu().numpy()
# Applies a guard condition to handle missing data and enforce expected preconditions.
if kpts.shape[0] < 17:
    # Executes this step as part of the end to end video pose analytics workflow.
    continue

# Executes this step as part of the end to end video pose analytics workflow.
s = ready_score_from_kpts(kpts)
# Applies a guard condition to handle missing data and enforce expected preconditions.
if np.isfinite(s):
    # Executes this step as part of the end to end video pose analytics workflow.
    scores.append(float(s))

# Executes this step as part of the end to end video pose analytics workflow.
scores = np.array(scores, dtype=float)
# Returns the computed value to the caller for downstream use.
return {
# Executes this step as part of the end to end video pose analytics workflow.
"target_id": int(target_id) if target_id is not None else None,










# Executes this step as part of the end to end video pose analytics workflow.
"frames_scored": int(len(scores)),
# Executes this step as part of the end to end video pose analytics workflow.
"mean": float(scores.mean()) if len(scores) else None,
# Executes this step as part of the end to end video pose analytics workflow.
"p10": float(np.percentile(scores, 10)) if len(scores) else None,
# Executes this step as part of the end to end video pose analytics workflow.
"p50": float(np.percentile(scores, 50)) if len(scores) else None,
# Executes this step as part of the end to end video pose analytics workflow.
"p90": float(np.percentile(scores, 90)) if len(scores) else None,
# Executes this step as part of the end to end video pose analytics workflow.
}

In [ ]:
# Executes this step as part of the end to end video pose analytics workflow.
am_summary = summarize_video_ready_score_locked(AM_CLIP, model, AM_TARGET_BBOX, conf=CONF)
# Executes this step as part of the end to end video pose analytics workflow.
pro_summary = summarize_video_ready_score_locked(PRO_CLIP, model, PRO_TARGET_BBOX, conf=CONF)
# Logs key intermediate outputs to support debugging and auditability.
print("AMATEUR:", am_summary)
# Logs key intermediate outputs to support debugging and auditability.
print("PRO: ", pro_summary)
# Executes this step as part of the end to end video pose analytics workflow.
AMATEUR: {'target_id': 2, 'frames_scored': 290, 'mean': 59.77885066317961,
# Executes this step as part of the end to end video pose analytics workflow.
'p10': 44.57815985148762, 'p50': 61.60942816634097, 'p90': 73.87396034756188}

# Executes this step as part of the end to end video pose analytics workflow.
PRO:   {'target_id': 80, 'frames_scored': 205, 'mean': 70.10498923059207,
# Executes this step as part of the end to end video pose analytics workflow.
'p10': 53.48829459813247, 'p50': 71.77249328983204, 'p90': 83.18279446004516}
# Executes this step as part of the end to end video pose analytics workflow.
frames_scored How many frames produced a valid score This is a data quality check, not a perfor-
# Executes this step as part of the end to end video pose analytics workflow.
mance metric

# Executes this step as part of the end to end video pose analytics workflow.
mean Average readiness across the clip This is your primary scalar metric
# Executes this step as part of the end to end video pose analytics workflow.
p10 Bottom-end readiness Captures worst posture moments

# Executes this step as part of the end to end video pose analytics workflow.
p50 (median) Typical posture during the clip Often more robust than the mean
# Executes this step as part of the end to end video pose analytics workflow.
p90 Best posture moments Shows peak form capability

In [ ]:
        # Defines write_overlay_raw_locked to encapsulate reusable logic for the pipeline.
        def write_overlay_raw_locked(
                # Executes this step as part of the end to end video pose analytics workflow.
                video_in,
                # Executes this step as part of the end to end video pose analytics workflow.
                video_out_raw,
                # Executes this step as part of the end to end video pose analytics workflow.
                model,
                # Executes this step as part of the end to end video pose analytics workflow.
                init_bbox_xyxy,
                # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                conf=0.25,
                # Stores a configuration constant to standardize inputs and outputs across runs.
                label="",

                # Stores a configuration constant to standardize inputs and outputs across runs.
                tracker_cfg="bytetrack.yaml",
                # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                init_search_frames=45,
                # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                reacquire_iou_thresh=0.25,
        # Executes this step as part of the end to end video pose analytics workflow.
        ):
                # Opens the input video stream to read metadata and frames.
                cap = cv2.VideoCapture(video_in)
                # Applies a guard condition to handle missing data and enforce expected preconditions.
                if not cap.isOpened():
                # Executes this step as part of the end to end video pose analytics workflow.
                raise RuntimeError(f"Could not open video: {video_in}")











                # Executes this step as part of the end to end video pose analytics workflow.
                fps = cap.get(cv2.CAP_PROP_FPS) or 30
                # Executes this step as part of the end to end video pose analytics workflow.
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                # Executes this step as part of the end to end video pose analytics workflow.
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                # Executes this step as part of the end to end video pose analytics workflow.
                cap.release()

                # Applies a guard condition to handle missing data and enforce expected preconditions.
                if os.path.exists(video_out_raw):
                # Executes this step as part of the end to end video pose analytics workflow.
                os.remove(video_out_raw)

                # Initializes a video writer to persist overlay frames as an output artifact.
                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                # Initializes a video writer to persist overlay frames as an output artifact.
                writer = cv2.VideoWriter(video_out_raw, fourcc, fps, (w, h))

                # Executes this step as part of the end to end video pose analytics workflow.
                init_bbox = np.array(init_bbox_xyxy, dtype=float)
                # Executes this step as part of the end to end video pose analytics workflow.
                target_id = None
                # Executes this step as part of the end to end video pose analytics workflow.
                last_bbox = None
                # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                frame_idx = 0


                # Runs multi object tracking with pose keypoints to generate frame level observations.
                for r in model.track(
                # Executes this step as part of the end to end video pose analytics workflow.
                source=video_in,
                # Executes this step as part of the end to end video pose analytics workflow.
                conf=conf,
                # Executes this step as part of the end to end video pose analytics workflow.
                stream=True,
                # Executes this step as part of the end to end video pose analytics workflow.
                persist=True,
                # Executes this step as part of the end to end video pose analytics workflow.
                tracker=tracker_cfg,
                # Executes this step as part of the end to end video pose analytics workflow.
                verbose=False,
                # Executes this step as part of the end to end video pose analytics workflow.
                ):
                # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                frame_idx += 1
                # Executes this step as part of the end to end video pose analytics workflow.
                frame = r.plot() # draws skeletons/boxes as Ultralytics sees them

                # Executes this step as part of the end to end video pose analytics workflow.
                score = None
                # Executes this step as part of the end to end video pose analytics workflow.
                tgt_bbox = None

                # Applies a guard condition to handle missing data and enforce expected preconditions.
                if r.boxes is not None and len(r.boxes) > 0 and r.keypoints is not None:
                    # Executes this step as part of the end to end video pose analytics workflow.
                    boxes = r.boxes.xyxy.cpu().numpy().astype(float)

                    # Applies a guard condition to handle missing data and enforce expected preconditions.
                    if getattr(r.boxes, "id", None) is not None:
                        # Executes this step as part of the end to end video pose analytics workflow.
                        ids = r.boxes.id.cpu().numpy().astype(int)

                        # Applies a guard condition to handle missing data and enforce expected preconditions.
                        if target_id is None and frame_idx <= init_search_frames:
                        # Executes this step as part of the end to end video pose analytics workflow.
                        ious = np.array([_iou_xyxy(init_bbox, b) for b in boxes], dtype=float)
                        # Executes this step as part of the end to end video pose analytics workflow.
                        best = int(np.argmax(ious))
                        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                        if ious[best] >= 0.10:
                            # Executes this step as part of the end to end video pose analytics workflow.
                            target_id = int(ids[best])
                            # Executes this step as part of the end to end video pose analytics workflow.
                            last_bbox = boxes[best].copy()

                        # Executes this step as part of the end to end video pose analytics workflow.
                        idxs = np.where(ids == target_id)[0] if target_id is not None else np.array([], dtype=int)










                        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                        if target_id is not None and len(idxs) == 0 and last_bbox is not None:
                        # Executes this step as part of the end to end video pose analytics workflow.
                        ious = np.array([_iou_xyxy(last_bbox, b) for b in boxes], dtype=float)
                        # Executes this step as part of the end to end video pose analytics workflow.
                        best = int(np.argmax(ious))
                        # Applies a guard condition to handle missing data and enforce expected preconditions.
                        if ious[best] >= reacquire_iou_thresh:
                            # Executes this step as part of the end to end video pose analytics workflow.
                            target_id = int(ids[best])
                            # Executes this step as part of the end to end video pose analytics workflow.
                            idxs = np.array([best], dtype=int)
                        # Applies a guard condition to handle missing data and enforce expected preconditions.
                        if len(idxs) > 0:
                        # Executes this step as part of the end to end video pose analytics workflow.
                        i = int(idxs[0])
                        # Executes this step as part of the end to end video pose analytics workflow.
                        last_bbox = boxes[i].copy()
                        # Executes this step as part of the end to end video pose analytics workflow.
                        tgt_bbox = last_bbox.copy()
                        # Executes this step as part of the end to end video pose analytics workflow.
                        kpts = r.keypoints.xy[i].cpu().numpy()
                        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                        if kpts.shape[0] >= 17:
                            # Executes this step as part of the end to end video pose analytics workflow.
                            score = ready_score_from_kpts(kpts)

                    # Executes this step as part of the end to end video pose analytics workflow.
                    else:
                        # Executes this step as part of the end to end video pose analytics workflow.
                        ious = np.array([_iou_xyxy(init_bbox, b) for b in boxes], dtype=float)
                        # Executes this step as part of the end to end video pose analytics workflow.
                        i = int(np.argmax(ious))
                        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                        if ious[i] >= 0.05:
                        # Executes this step as part of the end to end video pose analytics workflow.
                        tgt_bbox = boxes[i].copy()
                        # Executes this step as part of the end to end video pose analytics workflow.
                        kpts = r.keypoints.xy[i].cpu().numpy()
                        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                        if kpts.shape[0] >= 17:
                            # Executes this step as part of the end to end video pose analytics workflow.
                            score = ready_score_from_kpts(kpts)
                # Applies a guard condition to handle missing data and enforce expected preconditions.
                if label:
                    # Executes this step as part of the end to end video pose analytics workflow.
                    cv2.putText(frame, label, (20, 40),
                            # Executes this step as part of the end to end video pose analytics workflow.
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                            # Executes this step as part of the end to end video pose analytics workflow.
                            (255, 255, 255), 2, cv2.LINE_AA)

                # Applies a guard condition to handle missing data and enforce expected preconditions.
                if tgt_bbox is not None:
                    # Executes this step as part of the end to end video pose analytics workflow.
                    x1, y1, x2, y2 = map(int, tgt_bbox)

                    # Executes this step as part of the end to end video pose analytics workflow.
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
                    # Executes this step as part of the end to end video pose analytics workflow.
                    cv2.putText(frame, "TARGET", (x1, max(0, y1 - 10)),
                            # Executes this step as part of the end to end video pose analytics workflow.
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9,
                            # Executes this step as part of the end to end video pose analytics workflow.
                            (0, 255, 0), 2, cv2.LINE_AA)

                # Executes this step as part of the end to end video pose analytics workflow.
                txt = f"ReadyScore: {score:5.1f}" if (score is not None and np.
            # Executes this step as part of the end to end video pose analytics workflow.
            isfinite(score)) else "ReadyScore: n/a"
                # Executes this step as part of the end to end video pose analytics workflow.
                cv2.putText(frame, txt, (20, 85),
                        # Executes this step as part of the end to end video pose analytics workflow.
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                        # Executes this step as part of the end to end video pose analytics workflow.
                        (255, 255, 255), 2, cv2.LINE_AA)
                # Executes this step as part of the end to end video pose analytics workflow.
                writer.write(frame)











                # Executes this step as part of the end to end video pose analytics workflow.
                writer.release()
                # Returns the computed value to the caller for downstream use.
                return True

        # Defines convert_overlay_to_h264_faststart to encapsulate reusable logic for the pipeline.
        def convert_overlay_to_h264_faststart(inp, outp):
                # Applies a guard condition to handle missing data and enforce expected preconditions.
                if os.path.exists(outp):
                # Executes this step as part of the end to end video pose analytics workflow.
                os.remove(outp)
                # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                cmd = [
                # Builds an ffmpeg command that standardizes clip duration, frame rate, and resolution for inference.
                "ffmpeg", "-hide_banner", "-y",
                # Executes this step as part of the end to end video pose analytics workflow.
                "-i", inp,
                # Executes this step as part of the end to end video pose analytics workflow.
                "-c:v", "libx264", "-pix_fmt", "yuv420p",
                # Executes this step as part of the end to end video pose analytics workflow.
                "-preset", "veryfast", "-crf", "23",
                # Executes this step as part of the end to end video pose analytics workflow.
                "-movflags", "+faststart",
                # Executes this step as part of the end to end video pose analytics workflow.
                "-an",
                # Executes this step as part of the end to end video pose analytics workflow.
                outp
                # Executes this step as part of the end to end video pose analytics workflow.
                ]

                # Executes an external command to perform system level media or data operations.
                run_cmd(cmd)

        # Imports required packages to support reproducible data processing and model inference.
        import os

        # Applies a guard condition to handle missing data and enforce expected preconditions.
        if "RUN_DIR" not in globals():
                # Stores a configuration constant to standardize inputs and outputs across runs.
                RUN_DIR = "runs_ready_score"

        # Creates the local output directory to ensure downstream file writes succeed.
        os.makedirs(RUN_DIR, exist_ok=True)

        # Constructs a platform safe file path for intermediate artifacts.
        AM_OVERLAY_RAW = os.path.join(RUN_DIR, "amateur_clip1_overlay_raw.mp4")
        # Constructs a platform safe file path for intermediate artifacts.
        AM_OVERLAY_RAW_2 = os.path.join(RUN_DIR, "amateur_clip2_overlay_raw.mp4")
        # Constructs a platform safe file path for intermediate artifacts.
        AM_OVERLAY_RAW_3 = os.path.join(RUN_DIR, "amateur_clip3_overlay_raw.mp4")

        # Constructs a platform safe file path for intermediate artifacts.
        PRO_OVERLAY_RAW = os.path.join(RUN_DIR, "pro_clip1_overlay_raw.mp4")
        # Constructs a platform safe file path for intermediate artifacts.
        PRO_OVERLAY_RAW_2 = os.path.join(RUN_DIR, "pro_clip2_overlay_raw.mp4")
        # Constructs a platform safe file path for intermediate artifacts.
        PRO_OVERLAY_RAW_3 = os.path.join(RUN_DIR, "pro_clip3_overlay_raw.mp4")

        # Constructs a platform safe file path for intermediate artifacts.
        AM_OVERLAY_WEB = os.path.join(RUN_DIR, "amateur_clip1_overlay_h264.mp4")
        # Constructs a platform safe file path for intermediate artifacts.
        AM_OVERLAY_WEB_2 = os.path.join(RUN_DIR, "amateur_clip2_overlay_h264.mp4")

        # Constructs a platform safe file path for intermediate artifacts.
        AM_OVERLAY_WEB_3 = os.path.join(RUN_DIR, "amateur_clip3_overlay_h264.mp4")

        # Constructs a platform safe file path for intermediate artifacts.
        PRO_OVERLAY_WEB = os.path.join(RUN_DIR, "pro_clip1_overlay_h264.mp4")
        # Constructs a platform safe file path for intermediate artifacts.
        PRO_OVERLAY_WEB_2 = os.path.join(RUN_DIR, "pro_clip2_overlay_h264.mp4")
        # Constructs a platform safe file path for intermediate artifacts.
        PRO_OVERLAY_WEB_3 = os.path.join(RUN_DIR, "pro_clip3_overlay_h264.mp4")











        # Logs key intermediate outputs to support debugging and auditability.
        print("Overlay path variables defined.")

        # Executes this step as part of the end to end video pose analytics workflow.
        Overlay path variables defined.

In [ ]:
# Stores a configuration constant to standardize inputs and outputs across runs.
write_overlay_raw_locked(AM_CLIP, AM_OVERLAY_RAW, model, AM_TARGET_BBOX, conf=CONF, label="AMATEUR 1 (10s)")
# Stores a configuration constant to standardize inputs and outputs across runs.
write_overlay_raw_locked(AM_CLIP_2, AM_OVERLAY_RAW_2, model, AM_TARGET_BBOX_2, conf=CONF, label="AMATEUR 2 (10s)")
# Stores a configuration constant to standardize inputs and outputs across runs.
write_overlay_raw_locked(AM_CLIP_3, AM_OVERLAY_RAW_3, model, AM_TARGET_BBOX_3, conf=CONF, label="AMATEUR 3 (10s)")
# Stores a configuration constant to standardize inputs and outputs across runs.
write_overlay_raw_locked(PRO_CLIP, PRO_OVERLAY_RAW, model, PRO_TARGET_BBOX, conf=CONF, label="PRO 1 (10s)")
# Stores a configuration constant to standardize inputs and outputs across runs.
write_overlay_raw_locked(PRO_CLIP_2, PRO_OVERLAY_RAW_2, model, PRO_TARGET_BBOX_2, conf=CONF, label="PRO 2 (10s)")
# Stores a configuration constant to standardize inputs and outputs across runs.
write_overlay_raw_locked(PRO_CLIP_3, PRO_OVERLAY_RAW_3, model, PRO_TARGET_BBOX_3, conf=CONF, label="PRO 3 (10s)")
# Executes this step as part of the end to end video pose analytics workflow.
convert_overlay_to_h264_faststart(AM_OVERLAY_RAW, AM_OVERLAY_WEB)
# Executes this step as part of the end to end video pose analytics workflow.
convert_overlay_to_h264_faststart(AM_OVERLAY_RAW_2, AM_OVERLAY_WEB_2)
# Executes this step as part of the end to end video pose analytics workflow.
convert_overlay_to_h264_faststart(AM_OVERLAY_RAW_3, AM_OVERLAY_WEB_3)
# Executes this step as part of the end to end video pose analytics workflow.
convert_overlay_to_h264_faststart(PRO_OVERLAY_RAW, PRO_OVERLAY_WEB)
# Executes this step as part of the end to end video pose analytics workflow.
convert_overlay_to_h264_faststart(PRO_OVERLAY_RAW_2, PRO_OVERLAY_WEB_2)
# Executes this step as part of the end to end video pose analytics workflow.
convert_overlay_to_h264_faststart(PRO_OVERLAY_RAW_3, PRO_OVERLAY_WEB_3)
# Logs key intermediate outputs to support debugging and auditability.
print("Overlays created:")
# Logs key intermediate outputs to support debugging and auditability.
print(AM_OVERLAY_WEB, AM_OVERLAY_WEB_2, AM_OVERLAY_WEB_3)
# Logs key intermediate outputs to support debugging and auditability.
print(PRO_OVERLAY_WEB, PRO_OVERLAY_WEB_2, PRO_OVERLAY_WEB_3)
# Executes this step as part of the end to end video pose analytics workflow.
Overlays created:
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/amateur_clip1_overlay_h264.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/amateur_clip2_overlay_h264.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/amateur_clip3_overlay_h264.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/pro_clip1_overlay_h264.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/pro_clip2_overlay_h264.mp4
# Executes this step as part of the end to end video pose analytics workflow.
runs_ready_score/pro_clip3_overlay_h264.mp4

In [ ]:
        # Defines _video_html to encapsulate reusable logic for the pipeline.
        def _video_html(path, width=520):
                # Applies a guard condition to handle missing data and enforce expected preconditions.
                if not os.path.exists(path):
                # Stores a configuration constant to standardize inputs and outputs across runs.
                return f"<div style='color:#b00'>Missing: {path}</div>"
                # Renders outputs inline to support rapid qualitative validation of results.
                return Video(path, embed=True, width=width)._repr_html_()

        # Defines show_overlay_pairs to encapsulate reusable logic for the pipeline.
        def show_overlay_pairs():










                # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
                pairs = [
                # Executes this step as part of the end to end video pose analytics workflow.
                ("Pair 1 (10s)", AM_OVERLAY_WEB, PRO_OVERLAY_WEB),
                # Executes this step as part of the end to end video pose analytics workflow.
                ("Pair 2 (10s)", AM_OVERLAY_WEB_2, PRO_OVERLAY_WEB_2),
                # Executes this step as part of the end to end video pose analytics workflow.
                ("Pair 3 (10s)", AM_OVERLAY_WEB_3, PRO_OVERLAY_WEB_3),
                # Executes this step as part of the end to end video pose analytics workflow.
                ]
                # Iterates through streamed results to accumulate frame level metrics.
                for title, am_path, pro_path in pairs:
                html = f"""
                <div style="margin: 10px 0 22px 0;">
                <div style="font-weight:700; font-size:16px; margin-bottom:6px;
            ">{title}</div>
                <div style="display:flex; gap:18px; align-items:flex-start; flex-wrap:
            wrap;">
                    <div style="min-width:540px;">
                        <div style="font-weight:600; margin:0 0 6px 0;">Amateur</div>
                        {_video_html(am_path, width=520)}
                    </div>
                    <div style="min-width:540px;">
                        <div style="font-weight:600; margin:0 0 6px 0;">Pro</div>
                        {_video_html(pro_path, width=520)}
                    </div>
                </div>
                </div>
                """
                # Renders outputs inline to support rapid qualitative validation of results.
                display(HTML(html))
        # Executes this step as part of the end to end video pose analytics workflow.
        show_overlay_pairs()

        # Executes this step as part of the end to end video pose analytics workflow.
        <IPython.core.display.HTML object>
        # Executes this step as part of the end to end video pose analytics workflow.
        <IPython.core.display.HTML object>

        # Executes this step as part of the end to end video pose analytics workflow.
        <IPython.core.display.HTML object>

In [ ]:
# Defines _safe_locked_summary to encapsulate reusable logic for the pipeline.
def _safe_locked_summary(path, bbox):
    # Applies a guard condition to handle missing data and enforce expected preconditions.
    if not path or not os.path.exists(path):
        # Returns the computed value to the caller for downstream use.
        return {"frames_scored": 0, "mean": None, "p10": None, "p50": None, "p90": None, "target_id": None}
    # Returns the computed value to the caller for downstream use.
    return summarize_video_ready_score_locked(path, model, bbox, conf=CONF)
# Executes this step as part of the end to end video pose analytics workflow.
am_summary_1 = _safe_locked_summary(AM_CLIP, AM_TARGET_BBOX)
# Executes this step as part of the end to end video pose analytics workflow.
am_summary_2 = _safe_locked_summary(AM_CLIP_2, AM_TARGET_BBOX_2)
# Executes this step as part of the end to end video pose analytics workflow.
am_summary_3 = _safe_locked_summary(AM_CLIP_3, AM_TARGET_BBOX_3)

# Executes this step as part of the end to end video pose analytics workflow.
pro_summary_1 = _safe_locked_summary(PRO_CLIP, PRO_TARGET_BBOX)
# Executes this step as part of the end to end video pose analytics workflow.
pro_summary_2 = _safe_locked_summary(PRO_CLIP_2, PRO_TARGET_BBOX_2)
# Executes this step as part of the end to end video pose analytics workflow.
pro_summary_3 = _safe_locked_summary(PRO_CLIP_3, PRO_TARGET_BBOX_3)












# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
rows = [
    # Executes this step as part of the end to end video pose analytics workflow.
    {"clip_type":"amateur","clip_id":1,"clip_path":AM_CLIP, **am_summary_1},
    # Executes this step as part of the end to end video pose analytics workflow.
    {"clip_type":"amateur","clip_id":2,"clip_path":AM_CLIP_2, **am_summary_2},
    # Executes this step as part of the end to end video pose analytics workflow.
    {"clip_type":"amateur","clip_id":3,"clip_path":AM_CLIP_3, **am_summary_3},
    # Executes this step as part of the end to end video pose analytics workflow.
    {"clip_type":"pro","clip_id":1,"clip_path":PRO_CLIP, **pro_summary_1},
    # Executes this step as part of the end to end video pose analytics workflow.
    {"clip_type":"pro","clip_id":2,"clip_path":PRO_CLIP_2, **pro_summary_2},
    # Executes this step as part of the end to end video pose analytics workflow.
    {"clip_type":"pro","clip_id":3,"clip_path":PRO_CLIP_3, **pro_summary_3},
# Executes this step as part of the end to end video pose analytics workflow.
]

# Executes this step as part of the end to end video pose analytics workflow.
df = pd.DataFrame(rows)
# Executes this step as part of the end to end video pose analytics workflow.
df = df[["clip_type","clip_id","clip_path","target_id","frames_scored","mean","p10","p50","p90"]].
# Executes this step as part of the end to end video pose analytics workflow.
sort_values(["clip_type","clip_id"]).reset_index(drop=True)
# Renders outputs inline to support rapid qualitative validation of results.
display(df)
# Executes this step as part of the end to end video pose analytics workflow.
grouped = df.groupby("clip_type")[["mean","p10","p50","p90","frames_scored"]].
# Executes this step as part of the end to end video pose analytics workflow.
agg(["mean","std","min","max"])
# Logs key intermediate outputs to support debugging and auditability.
print("\nGrouped summary (ReadyScore: higher is better)")
# Renders outputs inline to support rapid qualitative validation of results.
display(grouped)
# Executes this step as part of the end to end video pose analytics workflow.
clip_type clip_id                  clip_path target_id \
# Executes this step as part of the end to end video pose analytics workflow.
0  amateur     1 runs_ready_score/amateur_clip1.mp4 466
# Executes this step as part of the end to end video pose analytics workflow.
1  amateur     2 runs_ready_score/amateur_clip2.mp4 549
# Executes this step as part of the end to end video pose analytics workflow.
2  amateur     3 runs_ready_score/amateur_clip3.mp4 581
# Executes this step as part of the end to end video pose analytics workflow.
3     pro      1    runs_ready_score/pro_clip1.mp4 655
# Executes this step as part of the end to end video pose analytics workflow.
4     pro      2    runs_ready_score/pro_clip2.mp4 687
# Executes this step as part of the end to end video pose analytics workflow.
5     pro      3    runs_ready_score/pro_clip3.mp4 800

# Executes this step as part of the end to end video pose analytics workflow.
frames_scored   mean      p10     p50      p90
# Executes this step as part of the end to end video pose analytics workflow.
0         282 60.204179 44.670534 61.810236 73.938388
# Executes this step as part of the end to end video pose analytics workflow.
1         193 70.079891 55.870137 70.428532 85.795101
# Executes this step as part of the end to end video pose analytics workflow.
2         263 57.275512 38.996841 58.315186 74.733034
# Executes this step as part of the end to end video pose analytics workflow.
3         205 70.104989 53.488295 71.772493 83.182794
# Executes this step as part of the end to end video pose analytics workflow.
4         300 61.425632 44.449437 62.898565 76.827614
# Executes this step as part of the end to end video pose analytics workflow.
5         177 72.373900 56.613542 73.624761 87.981999


# Executes this step as part of the end to end video pose analytics workflow.
Grouped summary (ReadyScore: higher is better)
            # Executes this step as part of the end to end video pose analytics workflow.
            mean                              p10         \
            # Executes this step as part of the end to end video pose analytics workflow.
            mean    std      min      max     mean    std

# Executes this step as part of the end to end video pose analytics workflow.
clip_type
# Executes this step as part of the end to end video pose analytics workflow.
amateur  62.519861 6.708936 57.275512 70.079891 46.512504 8.586132
# Executes this step as part of the end to end video pose analytics workflow.
pro      67.968174 5.778463 61.425632 72.373900 51.517091 6.317088

                                # Executes this step as part of the end to end video pose analytics workflow.
                                p50                          \
            # Executes this step as part of the end to end video pose analytics workflow.
            min      max     mean     std     min      max










# Executes this step as part of the end to end video pose analytics workflow.
clip_type
# Executes this step as part of the end to end video pose analytics workflow.
amateur  38.996841 55.870137 63.517985 6.234629 58.315186 70.428532
# Executes this step as part of the end to end video pose analytics workflow.
pro      44.449437 56.613542 69.431940 5.733364 62.898565 73.624761

            # Executes this step as part of the end to end video pose analytics workflow.
            p90                          frames_scored       \
            # Executes this step as part of the end to end video pose analytics workflow.
            mean    std      min      max       mean     std
# Executes this step as part of the end to end video pose analytics workflow.
clip_type
# Executes this step as part of the end to end video pose analytics workflow.
amateur  78.155508 6.628001 73.938388 85.795101 246.000000 46.872167
# Executes this step as part of the end to end video pose analytics workflow.
pro      82.664136 5.595251 76.827614 87.981999 227.333333 64.469631


        # Executes this step as part of the end to end video pose analytics workflow.
        min max
# Executes this step as part of the end to end video pose analytics workflow.
clip_type
# Executes this step as part of the end to end video pose analytics workflow.
amateur  193 282
# Executes this step as part of the end to end video pose analytics workflow.
pro      177 300

In [ ]:
md = """
**Interpreting the summary stats (ReadyScore):**
- **Higher is better** (100 = excellent “ready position”, 0 = poor readiness).
- **p90** = the player’s **best** readiness moments (top 10% of frames).
- **p50** = the typical / median frame.
- **p10** = the player’s **worst** readiness moments (bottom 10% of frames).
"""
# Renders outputs inline to support rapid qualitative validation of results.
display(HTML(md))

# Executes this step as part of the end to end video pose analytics workflow.
<IPython.core.display.HTML object>

In [ ]:
# Imports required packages to support reproducible data processing and model inference.
from openai import OpenAI

#if not os.environ.get("OPENAI_API_KEY"):
#   raise RuntimeError("OPENAI_API_KEY is not set. Set it in your environment before running this cell.")
# os.environ["OPENAI_API_KEY"] =
client = OpenAI()
# Executes this step as part of the end to end video pose analytics workflow.
pro_ref = grouped.loc["pro"]
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
pro_reference = {
    # Executes this step as part of the end to end video pose analytics workflow.
    "mean_ready_score": float(pro_ref[("mean", "mean")]),
    # Executes this step as part of the end to end video pose analytics workflow.
    "median_ready_score": float(pro_ref[("p50", "mean")]),
    # Executes this step as part of the end to end video pose analytics workflow.
    "low_end_ready_score": float(pro_ref[("p10", "mean")]), # worse tail










    # Executes this step as part of the end to end video pose analytics workflow.
    "high_end_ready_score": float(pro_ref[("p90", "mean")]), # best tail
# Executes this step as part of the end to end video pose analytics workflow.
}
# Logs key intermediate outputs to support debugging and auditability.
print("Pro reference:", pro_reference)

# Defines generate_feedback to encapsulate reusable logic for the pipeline.
def generate_feedback(amateur_row, pro_ref):
    # Defines fmt to encapsulate reusable logic for the pipeline.
    def fmt(x):
        # Implements error handling to surface failures with actionable diagnostics.
        try:
        # Applies a guard condition to handle missing data and enforce expected preconditions.
        if x is None:
            # Returns the computed value to the caller for downstream use.
            return "N/A"
        # Executes this step as part of the end to end video pose analytics workflow.
        x = float(x)
        # Applies a guard condition to handle missing data and enforce expected preconditions.
        if not np.isfinite(x):
            # Returns the computed value to the caller for downstream use.
            return "N/A"
        # Returns the computed value to the caller for downstream use.
        return f"{x:.1f}"
        # Implements error handling to surface failures with actionable diagnostics.
        except Exception:
        # Returns the computed value to the caller for downstream use.
        return "N/A"


    prompt = f"""
You are a pickleball coach providing technique feedback based on pose-derived metrics.
The metrics summarize a player's ready-position posture over a 10-second clip.
ReadyScore ranges 0–100, and HIGHER is BETTER (100 = excellent readiness).

Professional reference (average across clips):
- Mean ready score: {fmt(pro_ref['mean_ready_score'])}
- Median ready score: {fmt(pro_ref['median_ready_score'])}
- Low-end (p10, worse moments): {fmt(pro_ref['low_end_ready_score'])}
- High-end (p90, best moments): {fmt(pro_ref['high_end_ready_score'])}

Amateur clip metrics:
- Mean ready score: {fmt(amateur_row.get('mean'))}
- Median ready score: {fmt(amateur_row.get('p50'))}
- Low-end (p10, worse moments): {fmt(amateur_row.get('p10'))}
- High-end (p90, best moments): {fmt(amateur_row.get('p90'))}

Task:
1) Compare the amateur to the professional reference.
2) Identify 2–3 concrete posture or readiness issues (knee bend, stance width, 
forward lean, paddle/hand position).
3) Include one positive observation.
4) Suggest one actionable drill.
5) Keep feedback concise, coach-like, and specific.
"""
    # Executes this step as part of the end to end video pose analytics workflow.
    resp = client.chat.completions.create(
        # Stores a configuration constant to standardize inputs and outputs across runs.
        model="gpt-4o-mini",
        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        messages=[










        # Executes this step as part of the end to end video pose analytics workflow.
        {"role": "system", "content": "You are an expert pickleball coach.
# Executes this step as part of the end to end video pose analytics workflow.
"},
        # Executes this step as part of the end to end video pose analytics workflow.
        {"role": "user", "content": prompt},
        # Executes this step as part of the end to end video pose analytics workflow.
        ],
        # Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
        temperature=0.4,
    # Executes this step as part of the end to end video pose analytics workflow.
    )
    # Returns the computed value to the caller for downstream use.
    return resp.choices[0].message.content.strip()
# Sets a deterministic parameter value to control pipeline behavior and evaluation comparability.
amateur_feedback = {}
# Stores a configuration constant to standardize inputs and outputs across runs.
for _, row in df[df["clip_type"] == "amateur"].iterrows():
    # Executes this step as part of the end to end video pose analytics workflow.
    clip_id = row["clip_id"]
    # Executes this step as part of the end to end video pose analytics workflow.
    amateur_feedback[f"amateur_clip_{clip_id}"] = generate_feedback(row, pro_reference)

# Iterates through streamed results to accumulate frame level metrics.
for clip, text in amateur_feedback.items():
    # Logs key intermediate outputs to support debugging and auditability.
    print("\n" + "=" * 60)
    # Logs key intermediate outputs to support debugging and auditability.
    print(clip.upper())
    # Logs key intermediate outputs to support debugging and auditability.
    print(text)

# Executes this step as part of the end to end video pose analytics workflow.
Pro reference: {'mean_ready_score': 67.9681739277063, 'median_ready_score':
# Executes this step as part of the end to end video pose analytics workflow.
69.43193982633028, 'low_end_ready_score': 51.517091127633364,
# Executes this step as part of the end to end video pose analytics workflow.
'high_end_ready_score': 82.6641356294773}

# Executes this step as part of the end to end video pose analytics workflow.
============================================================
# Executes this step as part of the end to end video pose analytics workflow.
AMATEUR_CLIP_1

# Executes this step as part of the end to end video pose analytics workflow.
**1) Comparison:**
# Executes this step as part of the end to end video pose analytics workflow.
- Your mean ready score of **60.2** and median of **61.8** are below the
# Executes this step as part of the end to end video pose analytics workflow.
professional averages (68.0 and 69.4, respectively). This indicates that your
# Executes this step as part of the end to end video pose analytics workflow.
readiness in posture needs improvement.
# Executes this step as part of the end to end video pose analytics workflow.
- The low-end score of **44.7** suggests that there are moments where your
# Executes this step as part of the end to end video pose analytics workflow.
posture is significantly lacking compared to the professional standard.

# Executes this step as part of the end to end video pose analytics workflow.
**2) Posture/Readiness Issues:**
# Executes this step as part of the end to end video pose analytics workflow.
- **Knee Bend:** Your knee bend appears insufficient, which can limit your
# Executes this step as part of the end to end video pose analytics workflow.
ability to react quickly.
# Executes this step as part of the end to end video pose analytics workflow.
- **Stance Width:** Your stance may be too narrow, affecting your balance and
# Executes this step as part of the end to end video pose analytics workflow.
stability during play.
# Executes this step as part of the end to end video pose analytics workflow.
- **Paddle Position:** The paddle is often too low or not in a ready position,
# Executes this step as part of the end to end video pose analytics workflow.
which delays your response time to incoming shots.


# Executes this step as part of the end to end video pose analytics workflow.
**3) Positive Observation:**
# Executes this step as part of the end to end video pose analytics workflow.
- Your overall body alignment is good; you maintain a straight back, which is
# Executes this step as part of the end to end video pose analytics workflow.
crucial for effective movement.

# Executes this step as part of the end to end video pose analytics workflow.
**4) Actionable Drill:**










# Executes this step as part of the end to end video pose analytics workflow.
- **Ready Position Drill:** Practice the "Ready Position Shuffle." Start in your
# Executes this step as part of the end to end video pose analytics workflow.
ready position with knees bent, feet shoulder-width apart, and paddle at waist
# Executes this step as part of the end to end video pose analytics workflow.
height. Shuffle side to side while maintaining this posture for 30 seconds.
# Executes this step as part of the end to end video pose analytics workflow.
Focus on keeping your knees bent and stance wide. Repeat this drill several
# Executes this step as part of the end to end video pose analytics workflow.
times to reinforce muscle memory.

# Executes this step as part of the end to end video pose analytics workflow.
Stay focused on these areas, and you'll see improvement in your readiness on the
# Executes this step as part of the end to end video pose analytics workflow.
court!

# Executes this step as part of the end to end video pose analytics workflow.
============================================================
# Executes this step as part of the end to end video pose analytics workflow.
AMATEUR_CLIP_2

# Executes this step as part of the end to end video pose analytics workflow.
1. **Comparison to Professional Reference:**
# Executes this step as part of the end to end video pose analytics workflow.
- Your mean ready score of **70.1** is slightly above the professional
# Executes this step as part of the end to end video pose analytics workflow.
average of **68.0**, indicating a solid foundation in your readiness. However,
# Executes this step as part of the end to end video pose analytics workflow.
your median score of **70.4** is also higher than the professional median of
# Executes this step as part of the end to end video pose analytics workflow.
**69.4**, showing consistency in your readiness posture.


# Executes this step as part of the end to end video pose analytics workflow.
2. **Posture/Readiness Issues:**
# Executes this step as part of the end to end video pose analytics workflow.
- **Knee Bend:** Your knee bend appears insufficient at times, which can
# Executes this step as part of the end to end video pose analytics workflow.
hinder your ability to react quickly. Aim for a deeper bend to enhance agility.
# Executes this step as part of the end to end video pose analytics workflow.
- **Stance Width:** Your stance width is inconsistent; at times, it’s too
# Executes this step as part of the end to end video pose analytics workflow.
narrow, affecting your balance. A wider base will improve stability and
# Executes this step as part of the end to end video pose analytics workflow.
readiness.
# Executes this step as part of the end to end video pose analytics workflow.
- **Paddle Position:** Your paddle is often held too low, which can delay
# Executes this step as part of the end to end video pose analytics workflow.
your reaction time. Keep your paddle up and ready for quicker responses.

# Executes this step as part of the end to end video pose analytics workflow.
3. **Positive Observation:**
# Executes this step as part of the end to end video pose analytics workflow.
- Your forward lean is commendable, as it shows you are engaged and ready to
# Executes this step as part of the end to end video pose analytics workflow.
move toward the ball. This posture is crucial for effective court coverage.

# Executes this step as part of the end to end video pose analytics workflow.
4. **Actionable Drill:**
# Executes this step as part of the end to end video pose analytics workflow.
- **Ready Position Drill:** Practice the "Ready Position Flow" drill. Stand
# Executes this step as part of the end to end video pose analytics workflow.
in your ready position and focus on maintaining a deep knee bend and wide
# Executes this step as part of the end to end video pose analytics workflow.
stance. Alternate between moving side to side while keeping your paddle up and
# Executes this step as part of the end to end video pose analytics workflow.
ready. Do this for 5 minutes, emphasizing fluid transitions and maintaining
# Executes this step as part of the end to end video pose analytics workflow.
posture.

# Executes this step as part of the end to end video pose analytics workflow.
Keep up the good work, and focus on these areas to elevate your game!

# Executes this step as part of the end to end video pose analytics workflow.
============================================================

# Executes this step as part of the end to end video pose analytics workflow.
AMATEUR_CLIP_3

# Executes this step as part of the end to end video pose analytics workflow.
1. **Comparison to Professional Reference:**
# Executes this step as part of the end to end video pose analytics workflow.
- Your mean ready score of **57.3** is significantly below the professional
# Executes this step as part of the end to end video pose analytics workflow.
average of **68.0**, indicating room for improvement in your readiness.









# Executes this step as part of the end to end video pose analytics workflow.
- The median score of **58.3** also falls short of the professional median of
# Executes this step as part of the end to end video pose analytics workflow.
**69.4**, suggesting consistent issues in your posture.

# Executes this step as part of the end to end video pose analytics workflow.
2. **Posture/Readiness Issues:**
# Executes this step as part of the end to end video pose analytics workflow.
- **Knee Bend:** Your knee bend appears insufficient, which can limit your
# Executes this step as part of the end to end video pose analytics workflow.
ability to react quickly. Aim for a deeper bend to enhance your stability and
# Executes this step as part of the end to end video pose analytics workflow.
readiness.
# Executes this step as part of the end to end video pose analytics workflow.
- **Stance Width:** Your stance seems too narrow, reducing your balance. A
# Executes this step as part of the end to end video pose analytics workflow.
wider stance will provide better support and allow for quicker lateral
# Executes this step as part of the end to end video pose analytics workflow.
movements.
# Executes this step as part of the end to end video pose analytics workflow.
- **Forward Lean:** There is minimal forward lean in your posture. A slight
# Executes this step as part of the end to end video pose analytics workflow.
forward lean will help you stay engaged and ready to move in any direction.

# Executes this step as part of the end to end video pose analytics workflow.
3. **Positive Observation:**
# Executes this step as part of the end to end video pose analytics workflow.
- Your paddle position is generally good, staying close to your body and
# Executes this step as part of the end to end video pose analytics workflow.
ready for quick shots. This is a strong foundation to build upon.

# Executes this step as part of the end to end video pose analytics workflow.
4. **Actionable Drill:**
# Executes this step as part of the end to end video pose analytics workflow.
- **Ready Position Drill:** Practice the "Ready Position Hold" drill. Stand

# Executes this step as part of the end to end video pose analytics workflow.
in your ready position with a deep knee bend and wide stance. Hold this position
# Iterates through streamed results to accumulate frame level metrics.
for 30 seconds while focusing on maintaining a forward lean and keeping your
# Executes this step as part of the end to end video pose analytics workflow.
paddle up. Repeat this 5 times, gradually increasing the duration as you become
# Executes this step as part of the end to end video pose analytics workflow.
more comfortable.

# Executes this step as part of the end to end video pose analytics workflow.
Focus on these adjustments, and you’ll see improvements in your readiness on the
# Executes this step as part of the end to end video pose analytics workflow.
court! Keep up the hard work!

        # Executes this step as part of the end to end video pose analytics workflow.
        [ ]: